# Fine-tuning BERT on the discovered arXiv categories

Runs steps 4-6 of the assignment on a GPU, because 13 fine-tuning runs is not
feasible on CPU.

**What this notebook does and does not do.** Everything upstream — sampling,
splitting, embedding, clustering, and the choice of how many categories there are —
happened in the repository, under a fixed seed. This notebook consumes a single fixed
artifact (`trainset_*.parquet`) produced by `src/export_training_set.py`. It cannot
re-derive the categories, which is deliberate: the labels must be identical to the
ones the report analyses.

**The targets are discovered clusters, not arXiv's own labels.** The brief states the
number of categories is unknown, so categories are discovered first and the classifier
learns to assign an abstract to one of them. Macro-F1 here therefore measures how
learnable the discovered taxonomy is. It is comparable between runs on the same label
set and is *not* a score for the taxonomy itself — taxonomy quality is assessed
separately in the repository, by `stability.py` and `interpret.py`.

**Runtime:** Runtime -> Change runtime type -> T4 GPU.

## 1. Environment

In [ ]:
import subprocess, sys
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() or "NO GPU")

In [ ]:
# Pinned so the notebook behaves the same way on a later run.
!pip -q install "transformers>=4.40" "accelerate>=1.1" "datasets>=2.18" \
                "scikit-learn>=1.3" "pandas>=2.0" "pyarrow>=14"

## 2. The training set

Upload the `trainset_*.parquet` written by `src/export_training_set.py`, or put it in
Google Drive and set `PATH` to it. The companion `.json` manifest records the sha256,
the class counts and the split sizes, so the file used here can be matched to the one
the report describes.

In [ ]:
PHASE   = "main"              # "pilot" for a quick end-to-end check
LABELS  = "kmeans_k7"         # the cluster config exported, e.g. hdbscan_mcs100
NOISE   = "drop"              # "drop" or "class" — how to treat y == -1
EPOCHS  = 3

PATH = f"trainset_{PHASE}_minilm_{LABELS}.parquet"

import os
if not os.path.exists(PATH):
    from google.colab import files
    PATH = next(iter(files.upload()))
print("using", PATH)

In [ ]:
import numpy as np, pandas as pd

df = pd.read_parquet(PATH)
n_noise = int((df.y == -1).sum())

if NOISE == "drop":
    # An unassigned abstract carries no target, so it cannot supervise anything.
    # Dropping is honest; the cost is coverage below 100%, which is reported.
    df = df[df.y >= 0].reset_index(drop=True)
else:
    df["y"] = df.y.replace(-1, df.y.max() + 1)

classes = sorted(df.y.unique())
remap   = {c: i for i, c in enumerate(classes)}
df["label"] = df.y.map(remap)
NAMES = [f"cluster_{c}" for c in classes]

train = df[df.split == "train"].reset_index(drop=True)
val   = df[df.split == "val"].reset_index(drop=True)
test  = df[df.split == "test"].reset_index(drop=True)

print(f"{len(classes)} categories, {n_noise:,} noise abstracts ({NOISE})")
print(f"train={len(train):,}  val={len(val):,}  test={len(test):,}")
print(train.label.value_counts().sort_index().to_string())

## 3. One training run

Class imbalance is handled with weights in the loss rather than by resampling, so
every abstract is still seen exactly once per epoch and the sampling distribution
fixed in `preprocess.py` is left intact.

Macro-F1 is the headline metric and the model-selection criterion. Accuracy is
dominated by the largest cluster — the majority-class baseline reaches 48% accuracy on
the pilot while learning nothing.

In [ ]:
import torch, torch.nn.functional as F, time, json
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score)
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, EarlyStoppingCallback,
                          Trainer, TrainingArguments, set_seed)

ENCODERS = {
    "bert-base":  "bert-base-uncased",
    "distilbert": "distilbert-base-uncased",
    "scibert":    "allenai/scibert_scivocab_uncased",
}
MAX_LEN = 256   # fits 67.2% of abstracts whole; 128 would fit only 20.5%


class Abstracts(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tok):
        self.enc = tok(list(texts), truncation=True, max_length=MAX_LEN)
        self.labels = np.asarray(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = int(self.labels[i])
        return item


def metrics_fn(p):
    preds = np.argmax(p[0], axis=-1)
    return {"macro_f1": f1_score(p[1], preds, average="macro", zero_division=0),
            "accuracy": accuracy_score(p[1], preds)}


def weighted_loss(weights):
    def compute(outputs, labels, num_items_in_batch=None):
        return F.cross_entropy(
            outputs.logits.view(-1, outputs.logits.size(-1)), labels.view(-1),
            weight=weights.to(outputs.logits.device))
    return compute


def score(trainer, ds, split):
    out = trainer.predict(ds)
    preds, labels = np.argmax(out.predictions, axis=-1), out.label_ids
    rep = classification_report(labels, preds, target_names=NAMES,
                                output_dict=True, zero_division=0)
    print(f"\n{split}: macro-F1={rep['macro avg']['f1-score']:.4f}  "
          f"accuracy={rep['accuracy']:.4f}")
    return {"macro_f1": round(rep["macro avg"]["f1-score"], 4),
            "weighted_f1": round(rep["weighted avg"]["f1-score"], 4),
            "accuracy": round(rep["accuracy"], 4),
            "per_class": {n: {k: round(v, 4) for k, v in rep[n].items()}
                          for n in NAMES},
            "confusion_matrix": confusion_matrix(labels, preds).tolist()}


def run(encoder="bert-base", lr=2e-5, batch=16, epochs=EPOCHS, seed=42,
        weighted=True, quiet=True):
    set_seed(seed)
    name = ENCODERS[encoder]
    tag  = f"{encoder}_lr{lr:g}_bs{batch}_ep{epochs:g}_seed{seed}"
    tok  = AutoTokenizer.from_pretrained(name)
    model = AutoModelForSequenceClassification.from_pretrained(
        name, num_labels=len(classes),
        id2label={i: n for i, n in enumerate(NAMES)},
        label2id={n: i for i, n in enumerate(NAMES)})

    sets = {k: Abstracts(p.abstract, p.label, tok)
            for k, p in (("train", train), ("val", val), ("test", test))}

    counts = np.bincount(train.label, minlength=len(classes)).astype(float)
    w = len(train) / (len(classes) * np.maximum(counts, 1))
    w = torch.tensor(w / w.mean(), dtype=torch.float32)

    args = TrainingArguments(
        output_dir=f"/content/out/{tag}",
        learning_rate=lr,
        per_device_train_batch_size=batch,
        per_device_eval_batch_size=128,
        num_train_epochs=epochs,
        weight_decay=0.01,
        warmup_steps=int(0.1 * len(train) * epochs / batch),
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1", greater_is_better=True,
        fp16=torch.cuda.is_available(),
        logging_steps=200, seed=seed, report_to=[],
        disable_tqdm=quiet,
    )
    trainer = Trainer(
        model=model, args=args,
        train_dataset=sets["train"], eval_dataset=sets["val"],
        processing_class=tok, data_collator=DataCollatorWithPadding(tok),
        compute_metrics=metrics_fn,
        compute_loss_func=weighted_loss(w) if weighted else None,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    t0 = time.time()
    trainer.train()
    mins = (time.time() - t0) / 60

    res = {"tag": tag, "encoder": name, "learning_rate": lr, "batch_size": batch,
           "epochs": epochs, "seed": seed, "weighted_loss": weighted,
           "train_minutes": round(mins, 1),
           "val": score(trainer, sets["val"], "VALIDATION"),
           "test": score(trainer, sets["test"], "TEST")}
    print(f"[{tag}] val {res['val']['macro_f1']:.4f} / "
          f"test {res['test']['macro_f1']:.4f}  ({mins:.1f} min)")
    del model, trainer
    torch.cuda.empty_cache()
    return res

## 4. Baselines

The transformer is only worth its cost if it beats something cheap, and the majority
baseline shows why accuracy cannot be the headline metric.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline, make_union
from sklearn.svm import LinearSVC

baselines = {}

maj = DummyClassifier(strategy="most_frequent").fit(train.abstract, train.label)
p = maj.predict(val.abstract)
baselines["majority"] = {"macro_f1": round(f1_score(val.label, p, average="macro",
                                                    zero_division=0), 4),
                         "accuracy": round(accuracy_score(val.label, p), 4)}

svm = make_pipeline(
    make_union(
        TfidfVectorizer(sublinear_tf=True, min_df=3, ngram_range=(1, 2),
                        stop_words="english", max_features=300_000),
        TfidfVectorizer(sublinear_tf=True, min_df=3, analyzer="char_wb",
                        ngram_range=(3, 5), max_features=200_000)),
    LinearSVC(C=1.0, class_weight="balanced", random_state=42, dual="auto"),
).fit(train.abstract, train.label)
p = svm.predict(val.abstract)
baselines["tfidf_svm"] = {
    "macro_f1": round(f1_score(val.label, p, average="macro", zero_division=0), 4),
    "accuracy": round(accuracy_score(val.label, p), 4),
    "test_macro_f1": round(f1_score(test.label, svm.predict(test.abstract),
                                    average="macro", zero_division=0), 4)}

for k, v in baselines.items():
    print(f"{k:12} macro-F1={v['macro_f1']:.4f}  accuracy={v['accuracy']:.4f}")

## 5. Staged search

A full grid over learning rate x batch size x encoder x seed is 108 runs and almost
all of it is wasted. Four stages resolve one factor at a time, carrying the winner
forward: 13 runs instead of 108.

Selection uses **validation** macro-F1 only. Test is scored once per run and reported,
never compared during the search.

In [ ]:
runs = []

print("Stage 1 - learning rate")
for lr in [1e-5, 2e-5, 3e-5, 5e-5]:
    runs.append({**run(lr=lr), "stage": "lr"})
best_lr = max((r for r in runs if r["stage"] == "lr"),
              key=lambda r: r["val"]["macro_f1"])["learning_rate"]
print(f"-> lr={best_lr:g}")

In [ ]:
print("Stage 2 - batch size")
for bs in [8, 32]:        # 16 already measured in stage 1
    runs.append({**run(lr=best_lr, batch=bs), "stage": "batch"})
pool = [r for r in runs if r["learning_rate"] == best_lr]
best_bs = max(pool, key=lambda r: r["val"]["macro_f1"])["batch_size"]
print(f"-> batch={best_bs}")

In [ ]:
print("Stage 3 - encoder")
for enc in ["distilbert", "scibert"]:   # bert-base already measured
    runs.append({**run(encoder=enc, lr=best_lr, batch=best_bs), "stage": "encoder"})
best = max(runs, key=lambda r: r["val"]["macro_f1"])
best_enc = {v: k for k, v in ENCODERS.items()}[best["encoder"]]
print(f"-> {best_enc}")

In [ ]:
print("Stage 4 - seed spread")
for sd in [1337, 2024]:   # 42 already measured
    runs.append({**run(encoder=best_enc, lr=best_lr, batch=best_bs, seed=sd),
                 "stage": "seeds"})

## 6. Results

The seed spread matters: a single run implies a precision it does not have, and
Arcan (2026) reported no error bars at all.

In [ ]:
tab = pd.DataFrame([{
    "stage": r["stage"], "encoder": r["encoder"].split("/")[-1],
    "lr": r["learning_rate"], "batch": r["batch_size"], "seed": r["seed"],
    "val_macro_f1": r["val"]["macro_f1"], "test_macro_f1": r["test"]["macro_f1"],
    "test_accuracy": r["test"]["accuracy"], "minutes": r["train_minutes"],
} for r in runs]).sort_values("val_macro_f1", ascending=False)
display(tab)

seeds = [r for r in runs
         if (r["encoder"], r["learning_rate"], r["batch_size"])
         == (best["encoder"], best_lr, best_bs)]
spread = None
if len(seeds) > 1:
    f1 = [r["test"]["macro_f1"] for r in seeds]
    spread = {"seeds": [r["seed"] for r in seeds],
              "test_macro_f1_mean": round(float(np.mean(f1)), 4),
              "test_macro_f1_sd": round(float(np.std(f1, ddof=1)), 4)}
    print(f"\ntest macro-F1 {spread['test_macro_f1_mean']:.4f} "
          f"+/- {spread['test_macro_f1_sd']:.4f} over {len(f1)} seeds")

In [ ]:
summary = {
    "phase": PHASE, "label_config": LABELS, "noise_handling": NOISE,
    "n_categories": len(classes), "n_noise_abstracts": n_noise,
    "rows": {"train": len(train), "val": len(val), "test": len(test)},
    "selection_metric": "validation macro-F1; test never used for selection",
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "baselines": baselines,
    "best": {"encoder": best["encoder"], "learning_rate": best_lr,
             "batch_size": best_bs, "seed": best["seed"],
             "val_macro_f1": best["val"]["macro_f1"],
             "test_macro_f1": best["test"]["macro_f1"],
             "per_class": best["test"]["per_class"],
             "confusion_matrix": best["test"]["confusion_matrix"]},
    "seed_spread": spread,
    "runs": runs,
}
name = f"classifier_results_{PHASE}_{LABELS}.json"
with open(name, "w", encoding="utf-8") as fh:
    json.dump(summary, fh, indent=2)
tab.to_csv(f"tuning_{PHASE}_{LABELS}.csv", index=False)
print("wrote", name)

## 7. Bring the results back

Download both files and commit them to `reports/` in the repository. They are the
numbers the report quotes, so they belong under version control next to the code that
produced them.

In [ ]:
from google.colab import files
files.download(name)
files.download(f"tuning_{PHASE}_{LABELS}.csv")